In [ ]:
import pandas as pd
from datetime import datetime

path = 'input/PSLCClubTrialInformation.xlsx'

today = datetime.today().strftime('%Y-%m-%d')

trials = pd.read_excel(path)

def reorder_columns(df):
  return df[['Club', 'Teams', 'Start Date', 'End Date', 'Registration Deadline', 'Info', 'Contact']]

def append_last_updated(df, date):
  last_updated = pd.DataFrame({ 'Club': ['Last updated:'], 'Teams': [date]})
  return pd.concat([df, last_updated], ignore_index=True)

def add_socials_link(row):
  if isinstance(row['TrialInformation_TrialSocialPostLink'], str) and len(row['TrialInformation_TrialSocialPostLink']) > 0:
    return f" / <a href='{row['TrialInformation_TrialSocialPostLink']}' target=\"_blank\">Socials</a>"
  else:
    return ""

def map_teams(row):
  if row['League'] == 'Ontario Premier League':
    return row['TrialInformation_TeamDivision']
  elif row['League'] == 'British Columbia Premier League':
    return row['TrialInformation_TeamDivision2']
  elif row['League'] == 'Alberta Premier League':
    return row['TrialInformation_TeamDivision3']
  else:
    return row['TrialInformation_TeamDivision4']


In [ ]:
# Remove passed trials and order by date ascending

trials['End Date'] = pd.to_datetime(trials['TrialInformation_TrialPeriodEndDate'])

trials['Start Date'] = pd.to_datetime(trials['TrialInformation_TrialPeriodStartDate'])

trials['Registration Deadline'] = pd.to_datetime(trials['TrialInformation_PreregistrationOrDocumentSubmissionDeadline'])

trials = trials[trials['End Date'] >= today].sort_values(by='Start Date', ascending=True, ignore_index=True)

In [ ]:
# Rework columns to only have info I need

trials['Contact Name'] = trials["TrialInformation_ContactName_First"] + " " + trials["TrialInformation_ContactName_Last"]

trials['Contact Email'] = trials['TrialInformation_ContactEmail']

trials['Club'] = trials['PersonSubmittingForm_Club']

trials['League'] = trials['TrialInformation_InWhichPSLCProvincialLeagueDoesYourClubParticipateIn']

trials['Teams'] = trials.apply(lambda row: map_teams(row), axis=1)

trials['Contact'] = trials.apply(lambda row: f"<a href='mailto:{row['Contact Email']}'>{row['Contact Name']}</a>", axis=1)

trials['Info'] = trials.apply(lambda row: f"<a href='{row['TrialInformation_TrialWebsiteLink']}' target=\"_blank\">Website</a>{add_socials_link(row)}", axis=1)


In [ ]:
# Build PPL csv

ppl_trials = trials[trials['League'].str.contains('Prairies Premier League')]

ppl_trials = reorder_columns(ppl_trials)

ppl_trials = append_last_updated(ppl_trials, today)

ppl_trials.to_csv('output/ppl_trials.csv', index=False)

ppl_trials


In [ ]:
# Build OPL csv

opl_trials = trials[trials['League'].str.contains('Ontario Premier League')]

opl_trials = reorder_columns(opl_trials)

opl_trials = append_last_updated(opl_trials, today)

opl_trials.to_csv('output/opl_trials.csv', index=False)

opl_trials
